### ***Recurrent Neural Network(RNN)***
- ***A neural network designed for sequential data that maintains a hidden state to carry information from previous time steps.***

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/qa_dataset.csv")
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [2]:
df.shape

(90, 2)

In [3]:
## Tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [4]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [5]:
## vocab
vocab = {'<UNK>':0}

In [6]:
def build_vocab(row):
  tokenized_ques = tokenize(row['question'])
  tokenize_ans = tokenize(row['answer'])
  tokens = tokenized_ques + tokenize_ans

  for token in tokens:
    if token not in vocab:
      vocab[token] = len(vocab)

df.apply(build_vocab,axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [7]:
len(vocab)

324

In [8]:
# word --> numerical indices
def text_indices(text,vocab):
  idx_text = []
  for token in tokenize(text):
    if token in vocab:
      idx_text.append(vocab[token])
    else:
      idx_text.append(vocab['<UNK>'])
  return idx_text

In [9]:
text_indices("Who is Ankit Gupta", vocab)

[10, 2, 0, 0]

In [10]:
import torch
from torch.utils.data import Dataset,DataLoader

class QA_Dataset(Dataset):
  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return self.df.shape[0]

  def __getitem__(self, index):
    numeric_ques = text_indices(self.df.iloc[index]['question'],self.vocab)
    numeric_ans = text_indices(self.df.iloc[index]['answer'],self.vocab)
    return torch.tensor(numeric_ques),torch.tensor(numeric_ans)


In [11]:
dataset = QA_Dataset(df,vocab)
dataloader = DataLoader(dataset,batch_size=1,shuffle=True)

In [12]:
for ques,ans in dataloader:
  print(ques,ans[0])

tensor([[42, 86, 87, 88, 89, 39, 90]]) tensor([91])
tensor([[ 1,  2,  3,  4,  5, 73]]) tensor([74])
tensor([[ 42, 290, 291, 118, 292, 158, 293, 294]]) tensor([295])
tensor([[10, 29,  3, 30, 31]]) tensor([32])
tensor([[  1,   2,   3, 146, 147,  19, 148]]) tensor([149])
tensor([[ 42, 167,   2,   3,  17, 168, 169]]) tensor([170])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([36])
tensor([[ 42, 125,   2,  62,  63,   3, 126, 127]]) tensor([128])
tensor([[  1,   2,   3,  33,  34,   5, 245]]) tensor([246])
tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[  1,   2,   3, 234,   5, 235]]) tensor([131])
tensor([[ 10, 140,   3, 141, 142,  12, 143,  83,   3, 144]]) tensor([145])
tensor([[ 42,  86,  87, 241, 242,  19,  39, 243]]) tensor([244])
tensor([[ 1,  2,  3, 33, 34,  5, 35]]) tensor([36])
tensor([[ 10,  11, 189, 158, 190]]) tensor([191])
tensor([[ 10, 140,   3, 141, 171,   5,   3,  70, 172]]) tensor([173])
tensor([[ 1,  2,  3, 37, 38, 39, 40]]) tensor([41])
tensor([[10

In [13]:
# Build Our RNN Model
import torch.nn as nn
class RNN(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()

    self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
    self.rnn = nn.RNN(50,64,batch_first=True)
    self.fc = nn.Linear(64,vocab_size)

  def forward(self,ques):
    embedded_ques = self.embedding(ques)
    hidden,final = self.rnn(embedded_ques)
    output = self.fc(final.squeeze(0))
    return output

In [14]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))
print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [15]:
model = RNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),lr=0.001)

In [16]:
## Training Loop
epochs = 100
for epoch in range(epochs):
  total_loss = 0
  for ques,ans in dataloader:
    optimizer.zero_grad()

    # forward pass
    output = model(ques)

    # loss output shape (1,324) - (1)
    loss = criterion(output,ans[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()
    total_loss+=loss.item()
  print(f"Epoch:{epoch+1} & Loss:{total_loss:.4f}")

Epoch:1 & Loss:522.1703
Epoch:2 & Loss:456.8388
Epoch:3 & Loss:380.8559
Epoch:4 & Loss:313.5557
Epoch:5 & Loss:259.9011
Epoch:6 & Loss:210.0063
Epoch:7 & Loss:166.1619
Epoch:8 & Loss:127.6564
Epoch:9 & Loss:97.1618
Epoch:10 & Loss:73.8930
Epoch:11 & Loss:56.8514
Epoch:12 & Loss:44.4807
Epoch:13 & Loss:35.2400
Epoch:14 & Loss:28.7478
Epoch:15 & Loss:23.6350
Epoch:16 & Loss:19.7260
Epoch:17 & Loss:16.5858
Epoch:18 & Loss:14.1460
Epoch:19 & Loss:12.0662
Epoch:20 & Loss:10.4303
Epoch:21 & Loss:9.0525
Epoch:22 & Loss:7.9920
Epoch:23 & Loss:7.0613
Epoch:24 & Loss:6.3294
Epoch:25 & Loss:5.6636
Epoch:26 & Loss:5.1191
Epoch:27 & Loss:4.6399
Epoch:28 & Loss:4.2299
Epoch:29 & Loss:3.8423
Epoch:30 & Loss:3.5269
Epoch:31 & Loss:3.2401
Epoch:32 & Loss:2.9884
Epoch:33 & Loss:2.7597
Epoch:34 & Loss:2.5551
Epoch:35 & Loss:2.3686
Epoch:36 & Loss:2.2038
Epoch:37 & Loss:2.0523
Epoch:38 & Loss:1.9130
Epoch:39 & Loss:1.7860
Epoch:40 & Loss:1.6708
Epoch:41 & Loss:1.5653
Epoch:42 & Loss:1.4684
Epoch:43 & Loss

In [17]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")
  print(list(vocab.keys())[index])

In [18]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [19]:
list(vocab.keys())[7]

'paris'

In [20]:
predict(model, "Who painted the Mona Lisa?")

leonardo-da-vinci
